# Eksperimen Model ABSA dan NER untuk Review E-Commerce
Notebook ini berisi proses end-to-end dari persiapan data, preprocessing, pelatihan model Text Representation (TF-IDF), klasifikasi ABSA (Logistic Regression), dan ekstraksi entitas NER.

In [3]:
%pip install pandas numpy scikit-learn joblib

  Using cached pandas-3.0.3-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached scikit_learn-1.9.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached narwhals-2.23.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached pandas-3.0.3-cp311-cp311-win_amd64.whl (9.9 MB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB 640.0 kB/s eta 0:00:20
   ---------------------------------------- 0.2/12.6 MB 1.5 MB/s eta 0:00:09
   - -------------------------------------- 0.4/12.6 MB 3.1 MB/s eta 0:00:04
   -- ------------------------------------- 0.7/12.6 MB 3.6 MB/s eta 0:00:04
   --

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
import re
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. Load Data
# Pastikan path ini sesuai dengan struktur foldermu saat menjalankan notebook
df = pd.read_csv('../data/processed/balanced.csv')

# 2. Fungsi Preprocessing (Cleaning & Normalization)
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|@[^\s]+', '', text) # Hapus URL & Mention
    text = re.sub(r'[^a-z0-9\s]', '', text) # Hapus simbol
    return text

df['clean_text'] = df['review_text'].apply(preprocess_text)
display(df[['text', 'aspect', 'sentiment']].head())

KeyError: 'review_text'

In [ ]:
# Pisahkan Fitur (X) dan Target (y)
X = df['clean_text']
y = df['absa_label']

# Split data 80% Training, 20% Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Tahap Representasi Teks (TF-IDF)
tfidf = TfidfVectorizer(ngram_range=(1, 3), max_features=10000, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Tahap Pelatihan Model (Logistic Regression)
model_absa = LogisticRegression(class_weight='balanced', C=10.0, max_iter=2000, solver='lbfgs')
model_absa.fit(X_train_tfidf, y_train)

print("Training Selesai!")

In [ ]:
# Prediksi data testing
y_pred = model_absa.predict(X_test_tfidf)

# Hitung Metrik
print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Implementasi Named Entity Recognition (NER) dengan skema BIO
def rule_based_ner(text):
    aspects = ['harga', 'kualitas', 'pengiriman', 'pelayanan', 'kemasan']
    orgs = ['tokopedia', 'shopee', 'jne', 'jnt', 'sicepat']
    
    tokens = text.split()
    bio_tags = []
    
    for token in tokens:
        if token in aspects:
            bio_tags.append((token, "B-ASPECT"))
        elif token in orgs:
            bio_tags.append((token, "B-ORG"))
        else:
            bio_tags.append((token, "O"))
            
    return bio_tags

# Uji coba NER
teks_uji = "pengiriman jne sangat cepat harga murah"
print(f"Teks Asli: {teks_uji}")
print("Hasil NER:", rule_based_ner(teks_uji))

In [ ]:
# Simpan Vectorizer dan Model ke folder models/
joblib.dump(tfidf, '../models/bert/tfidf_vectorizer.pkl')
joblib.dump(model_absa, '../models/bert/absa_nb_model.pkl')
print("Model berhasil disimpan dan siap digunakan oleh Streamlit!")